# B2-022-probabilistic-latent-models — Practice p23 — Solution

**Type:** challenge · **Difficulty:** advanced · **Concepts:** gaussian-reparameterization, variational-autoencoder

*55 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import torch

x = torch.tensor([[1.0, 2.0, 0.0], [-1.0, 0.5, 1.5]])
x_hat = torch.tensor([[0.5, 1.0, 0.5], [0.0, 0.5, 1.0]])
mu = torch.tensor([[0.2, -0.4], [1.0, 0.0]])
logvar = torch.tensor([[0.5, 1.0], [0.2, 2.0]])
ATOL = 1e-6
RTOL = 1e-6

## Solution

### (B) The three defects

1. **Sign error** — `kl = -0.5 * (...)`. The Session 4 derivation gives $\mathrm{KL}(q\|p)=+\tfrac12\sum(\mu^2+\sigma^2-\log\sigma^2-1)\ge0$, and the negative ELBO **adds** it to the reconstruction; the minus sign rewards a large KL.
2. **Variance-versus-log-variance mix-up** — `logvar - torch.log(logvar)`. This treats the tensor as $\sigma^2$, but it holds $\ell=\log\sigma^2$; the derivation requires $\sigma^2-\log\sigma^2=e^{\ell}-\ell$, i.e. `logvar.exp() - logvar`.
3. **Reduction mismatch** — `((x - x_hat) ** 2).mean(dim=1)`. The Gaussian log-likelihood sums over features, $\tfrac12\sum_j(x_j-\hat x_j)^2$; averaging divides it by $d$ and silently multiplies the KL weight by $d$ (a hidden $\beta=d=3$ here).

In [ ]:
# (A) The teammate's code, verbatim, as a local variant
def buggy_negative_elbo(x, x_hat, mu, logvar):
    recon = 0.5 * ((x - x_hat) ** 2).mean(dim=1)
    kl = -0.5 * (mu ** 2 + logvar - torch.log(logvar) - 1.0).sum(dim=1)
    return (recon + kl).mean()


buggy_value = buggy_negative_elbo(x, x_hat, mu, logvar)


# (C) Repaired loss, p12 contract
def negative_elbo(x, x_hat, mu, logvar):
    for t in (x, x_hat, mu, logvar):
        if t.dim() != 2:
            raise ValueError("inputs must be 2-D")
    if x.shape != x_hat.shape or mu.shape != logvar.shape or x.shape[0] != mu.shape[0]:
        raise ValueError("shape mismatch")
    recon = 0.5 * ((x - x_hat) ** 2).sum(dim=1)
    kl = 0.5 * (mu ** 2 + logvar.exp() - logvar - 1.0).sum(dim=1)
    return (recon + kl).mean()


repaired = negative_elbo(x, x_hat, mu, logvar)


# (D) One defect at a time
def only_sign_defect(x, x_hat, mu, logvar):
    recon = 0.5 * ((x - x_hat) ** 2).sum(dim=1)
    kl = -0.5 * (mu ** 2 + logvar.exp() - logvar - 1.0).sum(dim=1)
    return (recon + kl).mean()


def only_variance_defect(x, x_hat, mu, logvar):
    recon = 0.5 * ((x - x_hat) ** 2).sum(dim=1)
    kl = 0.5 * (mu ** 2 + logvar - torch.log(logvar) - 1.0).sum(dim=1)
    return (recon + kl).mean()


def only_reduction_defect(x, x_hat, mu, logvar):
    recon = 0.5 * ((x - x_hat) ** 2).mean(dim=1)
    kl = 0.5 * (mu ** 2 + logvar.exp() - logvar - 1.0).sum(dim=1)
    return (recon + kl).mean()


single_defect_values = {
    "sign": only_sign_defect(x, x_hat, mu, logvar).item(),
    "variance": only_variance_defect(x, x_hat, mu, logvar).item(),
    "reduction": only_reduction_defect(x, x_hat, mu, logvar).item(),
}

mu_leaf = mu.clone().requires_grad_(True)
logvar_leaf = logvar.clone().requires_grad_(True)
negative_elbo(x, x_hat, mu_leaf, logvar_leaf).backward()
nan_demo = torch.log(torch.tensor([-0.5, 0.0]))     # what the buggy KL would hit for logvar <= 0
print(f"buggy value = {buggy_value.item():.7f}")
print(f"repaired value = {repaired.item():.7f}")
print(single_defect_values)

### (E) Why the variance mix-up produces `nan`, and why conventions must match

`torch.exp(0.5 * logvar)` is $e^{\ell/2}=\sqrt{e^{\ell}}=\sigma$: the reparameterization line treats the tensor as a **log-variance**, so any real value is legal, including negative ones (variance below 1).
The buggy KL computes `torch.log(logvar)`, i.e. it treats the same tensor as $\sigma^2$; for an entry $\le0$, $\log$ is `-inf` at $0$ and `nan` for negatives, so the loss (and its gradients) become `nan`/infinite as soon as the encoder outputs a variance below 1 — which is routine.
Even when every entry is positive, the KL then penalizes a distribution with variance $\ell$ while samples are drawn with variance $e^{\ell}$, so the regularizer pushes the wrong quantity toward 1 (its minimum is at $\ell=1$, i.e. actual variance $e$), and the objective is no longer the ELBO of the model being sampled.
Both lines must interpret `logvar` as $\log\sigma^2$: $z=\mu+e^{\ell/2}\varepsilon$ and $\mathrm{KL}=\tfrac12\sum(\mu^2+e^{\ell}-\ell-1)$.

### Answer check

In [ ]:
assert torch.allclose(repaired, torch.tensor(2.3068655), atol=ATOL, rtol=RTOL)
assert repaired.shape == ()
for name, value in single_defect_values.items():
    assert abs(value - repaired.item()) > 0.1, name
assert bool(torch.isfinite(buggy_value))
assert abs(buggy_value.item() - repaired.item()) > 0.1
assert mu_leaf.grad is not None and bool(torch.isfinite(mu_leaf.grad).all())
assert logvar_leaf.grad is not None and bool(torch.isfinite(logvar_leaf.grad).all())
assert torch.isnan(nan_demo[0]) and torch.isinf(nan_demo[1])